In [ ]:
from selenium import webdriver
from selenium_stealth import stealth
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
import time
import matplotlib.pyplot as plt
import seaborn as sns
import json
import numpy as np
import undetected_chromedriver as uc
import random
import os
import tempfile
from seleniumbase import Driver

In [ ]:
driver = Driver(uc=True, user_data_dir="my_perekrestok_profile")

url = "https://www.perekrestok.ru/cat/c/321/morozenoe"
driver.get(url)
time.sleep(3)

if "капча" in driver.page_source.lower() or "captcha" in driver.page_source.lower():
    input("Решите капчу в браузере и нажмите Enter: ")

html = driver.page_source
with open("perekrestok.html", "w", encoding="utf-8") as f:
    f.write(html)

driver.quit()

In [ ]:
with open("perekrestok.html", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

In [ ]:
categories = soup.find_all("div", class_="catalog-content-group__list")[1:9]
category_names = soup.find_all("h2", class_="catalog-content-group__title")[1:9]

In [ ]:
print(len(categories))
print(len(category_names))

In [ ]:
for i in range(8):
    print(category_names[i].text)

In [ ]:
category0 = categories[0]
url0 = f'https://www.perekrestok.ru{category0.find_all("a", class_="product-card__link")[0]["href"]}'
price0 = category0.find_all("div", class_="price-new")[0].text
name0 = category0.find_all("div", class_="product-card__title-wrapper")[0].text
rate0 = category0.find_all("div", class_="rating-value")[0].text
weight0 = category0.find_all("div", class_="product-card__size")[0].text

In [ ]:
driver = Driver(uc=True, user_data_dir="my_perekrestok_profile")

url = url0
driver.get(url)
time.sleep(3)

if "капча" in driver.page_source.lower() or "captcha" in driver.page_source.lower():
    input("Решите капчу в браузере и нажмите Enter: ")

html = driver.page_source

driver.quit()
soup0 = BeautifulSoup(html, "html.parser")

In [ ]:
n_reviews = soup0.find("a", class_="sc-gsTEea czKMGm product__review").find("span").text

In [ ]:
print(url0)
print(rate0)
print(price0)
print(name0)
print(n_reviews)
print(weight0)

In [ ]:
driver = Driver(uc=True, user_data_dir="my_perekrestok_profile")

data = []

for cat_idx, category in enumerate(categories):

    current_niche = category_names[cat_idx].text.strip()

    links = category.find_all("a", class_="product-card__link")
    prices = category.find_all("div", class_="price-new")
    names = category.find_all("div", class_="product-card__title-wrapper")
    ratings = category.find_all("div", class_="rating-value")
    weights = category.find_all("div", class_="product-card__size")

    items_count = min(len(links), len(prices), len(names), len(ratings), len(weights))

    for i in range(items_count):
        try:
            rate_raw = ratings[i].text.strip()
            if not rate_raw:
                continue

            if float(rate_raw.replace(",", ".")) < 4.5:
                continue

            name_raw = names[i].text.strip()
            price_raw = prices[i].text.strip()
            weight_raw = weights[i].text.strip()
            url = f"https://www.perekrestok.ru{links[i]['href']}"

            driver.get(url)

            time.sleep(random.uniform(3, 6))

            page_src = driver.page_source.lower()
            if (
                "капча" in page_src
                or "captcha" in page_src
                or "just a moment" in page_src
            ):
                input("Решите капчу в браузере и нажмите Enter: ")
                time.sleep(2)

            item_soup = BeautifulSoup(driver.page_source, "html.parser")
            review_elem = item_soup.find("a", class_="sc-gsTEea czKMGm product__review")
            reviews_raw = review_elem.text.strip() if review_elem else ""

            data.append(
                {
                    "Ниша": current_niche,
                    "Название": name_raw,
                    "Цена, ₽": price_raw,
                    "Вес, г": weight_raw,
                    "Рейтинг": rate_raw,
                    "Кол-во отзывов": reviews_raw,
                    "Ссылка": url,
                    "Цена за 100 г, ₽": None,
                }
            )

        except Exception as e:
            print(f"Ошибка (индекс {i}): {e}")
            continue


driver.quit()


df = pd.DataFrame(data)
df.head()

In [ ]:
df.to_csv("perek_morozhenoe.csv", index=False)

In [ ]:
CSV_PATH = "perek_morozhenoe.csv"

BASE_COLUMNS = [
    "Название",
    "Цена, ₽",
    "Вес, г",
    "Цена за 100 г, ₽",
    "Рейтинг",
    "Кол-во отзывов",
    "Ссылка",
]

PP_PATTERN = r"(?i)(?:протеин|без сахара|без сахарозы|без\s?лактоз|безлактоз)"


def _to_number(x):
    s = re.sub(r"[^\d.,]", "", str(x)).replace(",", ".")
    m = re.search(r"\d+(?:\.\d+)?", s)
    return float(m.group()) if m else np.nan


def build_df(csv_path: str = CSV_PATH) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    df["Цена, ₽"] = df["Цена, ₽"].apply(_to_number)
    df["Вес, г"] = df["Вес, г"].apply(_to_number)
    df["Рейтинг"] = df["Рейтинг"].apply(_to_number)

    df["Кол-во отзывов"] = (
        df["Кол-во отзывов"]
        .astype(str)
        .str.extract(r"(\d[\d\s]*)")[0]
        .str.replace(r"\s", "", regex=True)
    )
    df["Кол-во отзывов"] = pd.to_numeric(df["Кол-во отзывов"], errors="coerce").fillna(
        0
    )

    df["Цена за 100 г, ₽"] = ((df["Цена, ₽"] / df["Вес, г"]) * 100).round(2)

    df = df[df["Рейтинг"].notna()].reset_index(drop=True)

    df = df.drop_duplicates(subset=["Название"]).reset_index(drop=True)

    df["ПП"] = df["Название"].str.contains(PP_PATTERN, regex=True, na=False)

    niche_dummies = pd.get_dummies(df["Ниша"], dtype=bool)
    niche_cols = list(niche_dummies.columns)
    df = pd.concat([df.drop(columns=["Ниша"]), niche_dummies], axis=1)

    main_niche = np.where(df["ПП"], "ПП", df[niche_cols].idxmax(axis=1))

    med_price = df.groupby(main_niche)["Цена за 100 г, ₽"].transform("median")
    med_rev = df.groupby(main_niche)["Кол-во отзывов"].transform("median").replace(0, 1)

    df["Ценовая премия"] = (df["Цена за 100 г, ₽"] / med_price).round(3)
    df["Относительная популярность"] = (
        np.log1p(df["Кол-во отзывов"]) / np.log1p(med_rev)
    ).round(3)
    df["Score"] = (
        df["Ценовая премия"] * df["Относительная популярность"] * (df["Рейтинг"] / 5)
    ).round(3)

    score_cols = ["ПП", "Ценовая премия", "Относительная популярность", "Score"]
    df = df[BASE_COLUMNS + niche_cols + score_cols]
    df = df.sort_values("Score", ascending=False).reset_index(drop=True)

    df.attrs["niche_cols"] = niche_cols
    return df


df = build_df()
display(df.head(15))

In [ ]:
print("Строк:", len(df))
print("Колонки:", list(df.columns))
print()
print("Пропуски в ключевых полях:")
for c in [
    "Цена, ₽",
    "Вес, г",
    "Рейтинг",
    "Кол-во отзывов",
    "Цена за 100 г, ₽",
    "Score",
]:
    print(f"  {c:22s}: {df[c].isna().sum()} NaN")
print()
print("Диапазоны:")
print(f"  Рейтинг:           {df['Рейтинг'].min()} – {df['Рейтинг'].max()}")
print(
    f"  Цена за 100 г:     {df['Цена за 100 г, ₽'].min()} – {df['Цена за 100 г, ₽'].max()}"
)
print(
    f"  Кол-во отзывов:    {df['Кол-во отзывов'].min():.0f} – {df['Кол-во отзывов'].max():.0f}"
)
print(f"  Score:             {df['Score'].min()} – {df['Score'].max()}")
print()

niche_cols = df.attrs["niche_cols"]
print("Медианный Score по нишам:")
for n in niche_cols:
    sub = df[df[n]]
    if len(sub):
        print(f"  {n:25s}: {sub['Score'].median():.2f}  ({len(sub)} тов.)")
pp_sub = df[df["ПП"]]
if len(pp_sub):
    print(f"  {'ПП':25s}: {pp_sub['Score'].median():.2f}  ({len(pp_sub)} тов.)")

In [ ]:
df[["Название", "Ценовая премия", "Относительная популярность", "Score"]].head(15)

In [ ]:
sns.set_style("whitegrid")

top15 = df.sort_values("Score", ascending=False).head(15).reset_index(drop=True)
palette = sns.color_palette("tab20", n_colors=len(top15))

fig, ax = plt.subplots(figsize=(11, 7))

for i, row in top15.iterrows():
    ax.scatter(
        row["Относительная популярность"],
        row["Ценовая премия"],
        s=200,
        color=palette[i],
        edgecolor="white",
        linewidth=1.5,
        label=f"{i+1}. {row['Название'][:50]} (score {row['Score']:.2f})",
        zorder=3,
    )

ax.axhline(1, color="black", linestyle="--", alpha=0.4, linewidth=1)
ax.axvline(1, color="black", linestyle="--", alpha=0.4, linewidth=1)

ax.set_xlabel("Относительная популярность (>1 = популярнее медианы ниши)", fontsize=11)
ax.set_ylabel("Премия по цене (>1 = дороже медианы ниши)", fontsize=11)
ax.set_title("Топ-15 кандидатов", fontsize=14, fontweight="bold", pad=15)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),
    ncol=2,
    fontsize=9,
    frameon=False,
)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
df.iloc[15:40]

In [ ]:
premium_gems = df[
    (df["Ценовая премия"] > 1.4) & (df["Рейтинг"] > 4.8) & (df["Кол-во отзывов"] > 50)
].sort_values(by="Score", ascending=False)

display(
    premium_gems[
        ["Название", "Цена за 100 г, ₽", "Ценовая премия", "Кол-во отзывов", "Рейтинг"]
    ].head(10)
)

In [ ]:
pp_df = df[df["ПП"]].sort_values(by="Score", ascending=False)

display(
    pp_df[
        [
            "Название",
            "Цена за 100 г, ₽",
            "Ценовая премия",
            "Относительная популярность",
            "Рейтинг",
            "Score",
        ]
    ].head(20)
)